# GuitarSet slicer — corrected

## What was wrong

### 1. Filenames carried no recording ID (critical)

```python
unique_id = uuid.uuid4().hex[:8]
out_filepath = os.path.join(OUTPUT_DIR, f"{label}.{unique_id}.wav")
```

A random UUID per clip makes every clip its own group. With a 32 ms hop,
consecutive clips share 75% of their samples — but nothing in the filename says
which recording they came from, so there is no way to keep near-duplicates on
the same side of a train/val split. Measured inflation from that leak is up to
**+10.6 pp** for a model capable of memorising.

The fix keeps the label parse identical (`stem.split('.')[0]`) while embedding
the source recording:

```
60.00_BN1-129-Eb_comp_000042.wav
│  └──────── source id ───────┘└ clip index
└ label
```

### 2. No train/test split at all

Everything was written to one directory. Whatever produced your `training/` and
`testing/` folders afterwards split at the **clip** level, which contaminates the
test set as badly as the validation set. This version splits **by player**.

GuitarSet has 6 players (`00`–`05`). Holding out a whole player is stronger than
holding out recordings: it prevents the model from memorising one guitarist's
instrument, pick attack, and room. That is also GuitarSet's own convention.

### 3. Labels came from the window centre only

```python
t_mid = (start_sample + (window_samples / 2)) / SAMPLE_RATE
if note.time <= t_mid < (note.time + note.duration):
```

A window whose centre falls inside note B can still contain the tail of note A.
The clip then gets a single label for audio holding two pitches. At 128 ms this
happens about twice as often as at 64 ms. Now a note must cover essentially the
**whole** window, and no other note may overlap it anywhere.

### 4. `silence` was decided from annotations alone

A guitar keeps ringing after its annotated note ends, so windows labelled
`silence` frequently contained clearly audible notes. Now silence additionally
requires the actual audio to be below an RMS floor.

### 5. Unisons were discarded as chords

```python
elif len(active_notes) == 1:
```

The same pitch fretted on two strings gives `[60, 60]` — length 2, so it was
skipped as a chord. Counting distinct pitches fixes it.

### 6. Off-by-one dropped the last window

`range(0, len(y) - window_samples, hop)` stops one window early; it needs
`+ 1`.

### 7. Pitch bends were rounded to a wrong semitone

`round(note.value)` on a bent or vibrato'd note can land on the neighbouring
semitone while the audio sits between the two. Notes more than 25 cents off an
integer are now skipped.

## Also changed

`WINDOW_MS` is now **128** to match the training notebook, and the run ends with
a verification cell that re-parses the output with the *training notebook's own
logic* and confirms grouping works before you train on it.

In [ ]:
!pip install librosa soundfile jams numpy tqdm

## Configuration

In [ ]:
import os
import re
import glob
import shutil
import random
from collections import Counter, defaultdict

import numpy as np
import librosa
import soundfile as sf
import jams
from tqdm.notebook import tqdm

# ---- paths -----------------------------------------------------------------
AUDIO_DIR  = './audio_mono-mic'
JAMS_DIR   = './annotation'
OUTPUT_DIR = './ei_dataset'

# ---- geometry (must match the training notebook) ---------------------------
WINDOW_MS   = 128          # was 64. See the training notebook's intro for the
                           # measurement behind this.
HOP_MS      = 32           # matches the ESP32's 32 ms inference hop, so the
                           # model trains on every onset phase it will see live
SAMPLE_RATE = 16000

# ---- labelling strictness --------------------------------------------------
MIN_COVERAGE   = 0.95      # the labelled note must span >=95% of the window
MAX_CENTS_OFF  = 25        # skip bent/vibrato notes further than this from an
                           # integer semitone
SILENCE_RMS    = 0.004     # a "silence" window must ALSO be quieter than this
NOTE_MIN_RMS   = 0.010     # a labelled note window must be at least this loud,
                           # otherwise it is a decayed tail masquerading as a note

# ---- split -----------------------------------------------------------------
# GuitarSet filenames start with the player id: 00_BN1-129-Eb_comp_mic.wav
TEST_PLAYERS = {'05'}      # hold out one whole player. For 6-fold evaluation,
                           # rotate this through 00..05.

# ---- class selection -------------------------------------------------------
# Three filters, applied in this order. They do different jobs and none of them
# replaces the others.
#
# 1. RANGE (DROP_HIGHEST_N) -- a design decision, not a data one. It declares
#    the pitch span the product covers. The top of the guitar's range is played
#    rarely, so those classes are both scarce and out of scope.
#
# 2. EVIDENCE (MIN_*) -- a data decision. Catches any remaining class that is
#    too scarce to learn, to validate, or to measure. Rank-dropping alone cannot
#    do this job: it removes exactly N classes whether the tail is 6 long or 13.
#
# 3. CONTIGUITY -- keeps the surviving pitches a single unbroken run. Without
#    it, filter 2 could drop MIDI 55 while keeping 54 and 56, leaving a hole in
#    the middle of the model's range that is awkward to explain and awkward to
#    use.
DROP_HIGHEST_N    = 10     # drop the N highest MIDI classes outright.
                           # 0 disables. 'silence' is never affected.
ENFORCE_CONTIGUOUS = True  # keep the surviving pitches to one unbroken span
MAX_BRIDGE_GAP    = 2      # a run may bridge this many missing semitones. One
                           # scarce pitch mid-range then leaves a reported hole
                           # rather than truncating the whole range at it.

MIN_TRAIN_CLIPS   = 300    # a class needs this many training clips
MIN_TRAIN_SOURCES = 4      # ...from at least this many distinct recordings,
                           #    so a grouped 80/20 val split leaves >=3 / >=1
MIN_TEST_CLIPS    = 30     # ...and this many in the held-out player, or the
                           #    per-class test number is meaningless

# ---- balance ---------------------------------------------------------------
MAX_PER_CLASS = 2000       # cap per class per split, applied AFTER selection.
                           # With MIN_TRAIN_CLIPS=300 this bounds the imbalance
                           # ratio at 2000/300 = 6.7:1. Set to None to disable.
RANDOM_SEED = 42

window_samples = int(SAMPLE_RATE * WINDOW_MS / 1000.0)
hop_samples    = int(SAMPLE_RATE * HOP_MS / 1000.0)

print(f"window : {WINDOW_MS} ms = {window_samples} samples")
print(f"hop    : {HOP_MS} ms = {hop_samples} samples "
      f"({(1 - HOP_MS / WINDOW_MS) * 100:.0f}% overlap between neighbours)")
print(f"test   : player(s) {sorted(TEST_PLAYERS)} held out entirely")

## Slice

Each window is accepted only if exactly one distinct pitch overlaps it, that
pitch covers ≥95% of the window, nothing else overlaps, and the audio is loud
enough to actually contain the note. Everything else is counted and skipped, so
the tally at the end tells you *why* clips were rejected rather than leaving you
to guess.

In [ ]:
def player_of(file_id):
    """GuitarSet: '00_BN1-129-Eb_comp' -> '00'."""
    return file_id.split('_')[0]


def overlap(a0, a1, b0, b1):
    return max(0.0, min(a1, b1) - max(a0, b0))


for split in ('training', 'testing'):
    d = os.path.join(OUTPUT_DIR, split)
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

audio_files = sorted(glob.glob(os.path.join(AUDIO_DIR, '*.wav')))
print(f"Found {len(audio_files)} audio files.")

stats = Counter()
pending = defaultdict(list)      # (split, label) -> list of (path, audio)

for audio_path in tqdm(audio_files, desc="Slicing"):
    filename = os.path.basename(audio_path)
    file_id = filename.replace('_mic.wav', '').replace('.wav', '')

    jams_path = os.path.join(JAMS_DIR, f"{file_id}.jams")
    if not os.path.exists(jams_path):
        stats['file: no matching .jams'] += 1
        continue

    split = 'testing' if player_of(file_id) in TEST_PLAYERS else 'training'

    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    jam = jams.load(jams_path)

    # Flatten every string's notes into one list of (start, end, pitch).
    notes = []
    for anno in jam.search(namespace='note_midi'):
        for note in anno.data:
            notes.append((note.time, note.time + note.duration, float(note.value)))

    # +1 so the final full window is included (the original dropped it).
    for start in range(0, len(y) - window_samples + 1, hop_samples):
        t0 = start / SAMPLE_RATE
        t1 = (start + window_samples) / SAMPLE_RATE
        clip = y[start:start + window_samples]
        rms = float(np.sqrt(np.mean(clip ** 2)))

        hits = [(s, e, v) for (s, e, v) in notes if overlap(t0, t1, s, e) > 0.0]

        if not hits:
            # Annotation says nothing is sounding -- but the guitar may still be
            # ringing, so require the audio to be genuinely quiet too.
            if rms < SILENCE_RMS:
                label = 'silence'
            else:
                stats['skip: unannotated but audible'] += 1
                continue
        else:
            pitches = {round(v) for (_, _, v) in hits}
            if len(pitches) > 1:
                stats['skip: polyphonic'] += 1
                continue

            s, e, v = max(hits, key=lambda h: overlap(t0, t1, h[0], h[1]))

            if abs(v - round(v)) > MAX_CENTS_OFF / 100.0:
                stats['skip: pitch too far off a semitone (bend)'] += 1
                continue

            cov = overlap(t0, t1, s, e) / (t1 - t0)
            if cov < MIN_COVERAGE:
                stats['skip: note does not span the window'] += 1
                continue

            if rms < NOTE_MIN_RMS:
                stats['skip: labelled note too quiet (decayed tail)'] += 1
                continue

            label = str(round(v))

        idx = len(pending[(split, label)])
        # label . source_id _ index  -- keeps stem.split('.')[0] == label
        name = f"{label}.{file_id}_{idx:06d}.wav"
        pending[(split, label)].append((os.path.join(OUTPUT_DIR, split, name), clip))
        stats[f'kept: {split}'] += 1

print("\n--- outcomes ---")
for k, v in sorted(stats.items(), key=lambda kv: -kv[1]):
    print(f"  {v:>9,}  {k}")

## Select classes, balance, and write

Two separate operations, easy to confuse:

**Selection** removes classes there is not enough evidence for. GuitarSet's
pitch distribution has a long tail — the rarest class in a first pass had **5
clips from a single recording**. Such a class cannot be learned, cannot appear
in a grouped validation split, and cannot be measured on the held-out player.

**Balancing** then caps what remains. Note that capping alone cannot fix a bad
imbalance ratio: with a floor of 5 clips, reaching even 10:1 would require
`MAX_PER_CLASS = 50`, discarding ~96% of the data. The floor has to be raised by
dropping classes; the ceiling is what `MAX_PER_CLASS` controls.

A class is kept only if it clears the thresholds in **both** splits. Keeping a
class that exists in training but not in testing would leave the test set unable
to score it — and the training notebook would silently fold those files into
class 0.

Subsampling spreads across source recordings rather than truncating whichever
recording happened to be processed first.

In [ ]:
def src_of(path):
    """'60.00_BN1-129-Eb_comp_000042.wav' -> '60.00_BN1-129-Eb_comp'"""
    return os.path.basename(path).rsplit('_', 1)[0]


def label_sort_key(l):
    return (1, 0) if not l.isdigit() else (0, int(l))


counts = {s: {} for s in ('training', 'testing')}
sources = {s: {} for s in ('training', 'testing')}
for (split, label), items in pending.items():
    counts[split][label] = len(items)
    sources[split][label] = len({src_of(p) for p, _ in items})

all_labels = sorted(set(counts['training']) | set(counts['testing']),
                    key=label_sort_key)

numeric = sorted(int(l) for l in all_labels if l.isdigit())

# ---- filter 1: range -------------------------------------------------------
out_of_range = set()
if DROP_HIGHEST_N > 0 and numeric:
    out_of_range = {str(m) for m in numeric[-DROP_HIGHEST_N:]}
    print(f"range filter: dropping the {DROP_HIGHEST_N} highest MIDI classes "
          f"{sorted(out_of_range, key=int)}")
    print(f"              model range becomes MIDI {numeric[0]}-"
          f"{numeric[-DROP_HIGHEST_N - 1] if len(numeric) > DROP_HIGHEST_N else '?'}\n")

# ---- filter 2: evidence ----------------------------------------------------
rows = []
for l in all_labels:
    tr_c = counts['training'].get(l, 0)
    tr_s = sources['training'].get(l, 0)
    te_c = counts['testing'].get(l, 0)

    reasons = []
    if l in out_of_range:
        reasons.append("out of range")
    if tr_c < MIN_TRAIN_CLIPS:
        reasons.append(f"train clips {tr_c}<{MIN_TRAIN_CLIPS}")
    if tr_s < MIN_TRAIN_SOURCES:
        reasons.append(f"train sources {tr_s}<{MIN_TRAIN_SOURCES}")
    if te_c < MIN_TEST_CLIPS:
        reasons.append(f"test clips {te_c}<{MIN_TEST_CLIPS}")
    rows.append((l, tr_c, tr_s, te_c, reasons))

survivors = {l for l, _, _, _, r in rows if not r}

# ---- filter 3: contiguity --------------------------------------------------
# A pitch classifier should cover an unbroken span. But a single scarce pitch in
# the middle of the range must not cost us everything on one side of it, so runs
# are allowed to bridge gaps of up to MAX_BRIDGE_GAP semitones. Anything wider
# is a genuine break in the range and truncates it.
non_contig = set()
holes = []
if ENFORCE_CONTIGUOUS:
    nums = sorted(int(l) for l in survivors if l.isdigit())
    if nums:
        runs, cur = [], [nums[0]]
        for a, b in zip(nums, nums[1:]):
            if b - a - 1 <= MAX_BRIDGE_GAP:
                cur.append(b)
            else:
                runs.append(cur)
                cur = [b]
        runs.append(cur)

        best = max(runs, key=len)
        best_set = set(best)
        non_contig = {str(m) for m in nums if m not in best_set}
        holes = [m for m in range(best[0], best[-1] + 1) if m not in best_set]

        if non_contig:
            print(f"contiguity: survivors split into {len(runs)} runs separated "
                  f"by more than {MAX_BRIDGE_GAP} semitone(s).")
            print(f"            keeping MIDI {best[0]}-{best[-1]}, dropping "
                  f"{sorted(non_contig, key=int)}")
        if holes:
            print(f"contiguity: MIDI {holes} lack enough data but sit inside the "
                  f"kept range.")
            print(f"            The model will have {len(holes)} hole(s) in its "
                  f"span and can never predict those pitches.")
            print(f"            Lower MIN_TRAIN_CLIPS if you need them covered.")
        if non_contig or holes:
            print()
        survivors -= non_contig

# ---- report ----------------------------------------------------------------
print(f"{'class':>8} {'train':>8} {'srcs':>6} {'test':>7}   status")
print("-" * 76)
for l, tr_c, tr_s, te_c, reasons in rows:
    if l in non_contig:
        reasons = reasons + ["outside the contiguous run"]
    status = "keep" if not reasons else "DROP: " + "; ".join(reasons)
    print(f"{l:>8} {tr_c:>8,} {tr_s:>6} {te_c:>7,}   {status}")

keep_labels = survivors
kept_nums = sorted(int(l) for l in keep_labels if l.isdigit())
print(f"\nkeeping {len(keep_labels)} of {len(all_labels)} classes"
      + (f"  (MIDI {kept_nums[0]}-{kept_nums[-1]}"
         + (", contiguous" if kept_nums == list(range(kept_nums[0], kept_nums[-1] + 1))
            else ", WITH GAPS") + ")" if kept_nums else ""))
if not keep_labels:
    raise SystemExit("No class met the thresholds -- lower them and re-run.")

# ---- balance and write ----------------------------------------------------
rng = random.Random(RANDOM_SEED)
written = Counter()

for split in ('training', 'testing'):
    d = os.path.join(OUTPUT_DIR, split)
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

targets = [(k, v) for k, v in sorted(pending.items()) if k[1] in keep_labels]

for (split, label), items in tqdm(targets, desc="Writing"):
    if MAX_PER_CLASS is not None and len(items) > MAX_PER_CLASS:
        by_src = defaultdict(list)
        for it in items:
            by_src[src_of(it[0])].append(it)
        for v in by_src.values():
            rng.shuffle(v)
        picked, srcs = [], list(by_src.values())
        i = 0
        while len(picked) < MAX_PER_CLASS and any(srcs):
            bucket = srcs[i % len(srcs)]
            if bucket:
                picked.append(bucket.pop())
            else:
                srcs = [s for s in srcs if s]
                if not srcs:
                    break
                continue
            i += 1
        items = picked

    for path, clip in items:
        # PCM_16 so tf.audio.decode_wav reads it back as float in [-1, 1].
        sf.write(path, np.clip(clip, -1.0, 1.0), SAMPLE_RATE, subtype='PCM_16')
        written[(split, label)] += 1

print("\n--- written ---")
for split in ('training', 'testing'):
    vals = [v for (s, _), v in written.items() if s == split]
    if not vals:
        continue
    print(f"  {split:9s}: {sum(vals):>8,} clips, {len(vals)} classes, "
          f"min {min(vals):,} / max {max(vals):,} "
          f"(ratio {max(vals) / max(min(vals), 1):.1f}:1)")

## Verify before training

Re-parses the written files using the **training notebook's own** label and
source-id logic, and checks the things that are silent at training time:
grouping, class-set agreement between splits, imbalance, and source overlap.

In [ ]:
SOURCE_ID_REGEX = r'^(.+)_\d+$'      # same default as the training notebook

MAX_ACCEPTABLE_RATIO = 12.0


def parse(path):
    stem = os.path.basename(path).rsplit('.wav', 1)[0]
    m = re.match(SOURCE_ID_REGEX, stem)
    return stem.split('.')[0], (m.group(1) if m else stem)


ok = True
seen = {}

for split in ('training', 'testing'):
    paths = sorted(glob.glob(os.path.join(OUTPUT_DIR, split, '*.wav')))
    if not paths:
        print(f"\n=== {split} ===\n  FAIL: no files written."); ok = False; continue

    labels, groups = zip(*[parse(p) for p in paths])
    per_class = Counter(labels)
    src_per_class = defaultdict(set)
    for l, g in zip(labels, groups):
        src_per_class[l].add(g)
    min_src = min(len(v) for v in src_per_class.values())
    counts_sorted = sorted(per_class.values())
    ratio = counts_sorted[-1] / max(counts_sorted[0], 1)
    seen[split] = (set(labels), set(groups))

    print(f"\n=== {split} ===")
    print(f"  clips            : {len(paths):,}")
    print(f"  classes          : {len(per_class)}")
    print(f"  distinct sources : {len(set(groups))}")
    print(f"  sources per class: min {min_src}")
    print(f"  class balance    : min {counts_sorted[0]:,}  "
          f"median {counts_sorted[len(counts_sorted) // 2]:,}  "
          f"max {counts_sorted[-1]:,}  (ratio {ratio:.1f}:1)")

    if len(set(groups)) == len(paths):
        print("  FAIL: one source per clip -- grouping is broken."); ok = False
    elif split == 'training' and min_src < 3:
        print(f"  FAIL: a class has only {min_src} source(s); a grouped "
              "validation split would put whole classes on one side."); ok = False
    else:
        print("  OK: grouping works.")

    if ratio > MAX_ACCEPTABLE_RATIO:
        print(f"  FAIL: imbalance {ratio:.1f}:1 exceeds {MAX_ACCEPTABLE_RATIO}:1.")
        print("        Raise MIN_TRAIN_CLIPS (lifts the floor) before reaching")
        print("        for MAX_PER_CLASS -- capping cannot raise a floor.")
        ok = False
    else:
        print(f"  OK: imbalance {ratio:.1f}:1 is within tolerance.")

# ---- cross-split checks ----------------------------------------------------
if len(seen) == 2:
    tr_lab, tr_src = seen['training']
    te_lab, te_src = seen['testing']

    print(f"\nclass sets identical across splits: {tr_lab == te_lab}")
    if tr_lab != te_lab:
        if tr_lab - te_lab:
            print(f"  train-only: {sorted(tr_lab - te_lab)}")
        if te_lab - tr_lab:
            print(f"  test-only : {sorted(te_lab - tr_lab)}  <-- these would be")
            print("              silently folded into class 0 during training")
        print("  FAIL"); ok = False
    else:
        print("  OK")

    shared = tr_src & te_src
    print(f"\nsources in BOTH splits: {len(shared)}")
    if shared:
        print("  FAIL: test set is contaminated."); ok = False
    else:
        print("  OK: train and test share no source recording.")

print("\n" + ("ALL CHECKS PASSED -- safe to train."
              if ok else "PROBLEMS FOUND -- fix before training."))

## Notes

**Six-fold evaluation.** For a defensible number in the writeup, rotate
`TEST_PLAYERS` through `{'00'}` … `{'05'}`, re-slice, retrain, and report the
mean and spread. Holding out a single fixed player gives one sample of a
noticeably variable quantity — guitarists differ.

**If too much gets skipped.** The tally after slicing tells you which rule is
binding. In fast solo passages `note does not span the window` will dominate,
because a 128 ms window is longer than many played notes — that is the real
cost of the longer window, and it is worth seeing explicitly. Lowering
`MIN_COVERAGE` to ~0.8 recovers clips at the price of some boundary
contamination.

**Sanity-check by ear.** Play a handful of `silence.*.wav` files. If you hear
guitar, raise `SILENCE_RMS`. That class is the one most likely to be quietly
wrong, and it is the class the ESP32 sits in most of the time.